In [2]:
import sqlite3

conn = sqlite3.connect("mutual_fund_dw.db")
cursor = conn.cursor()

In [3]:
schema_sql = """

CREATE TABLE dim_scheme (
    scheme_id INTEGER PRIMARY KEY,
    scheme_name TEXT,
    category TEXT,
    fund_house TEXT,
    risk_level TEXT
);

CREATE TABLE dim_investor (
    investor_id INTEGER PRIMARY KEY,
    investor_name TEXT,
    city TEXT,
    state TEXT,
    investor_type TEXT
);

CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY,
    full_date DATE,
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER
);

CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY,
    investor_id INTEGER,
    scheme_id INTEGER,
    date_id INTEGER,
    transaction_type TEXT,
    units REAL,
    amount REAL,
    nav REAL,
    FOREIGN KEY (investor_id) REFERENCES dim_investor(investor_id),
    FOREIGN KEY (scheme_id) REFERENCES dim_scheme(scheme_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);

CREATE TABLE fact_nav_history (
    nav_record_id INTEGER PRIMARY KEY,
    scheme_id INTEGER,
    date_id INTEGER,
    nav_value REAL,
    aum REAL,
    FOREIGN KEY (scheme_id) REFERENCES dim_scheme(scheme_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);

CREATE TABLE fact_scheme_performance (
    performance_id INTEGER PRIMARY KEY,
    scheme_id INTEGER,
    date_id INTEGER,
    returns_1y REAL,
    returns_3y REAL,
    returns_5y REAL,
    benchmark_return REAL,
    FOREIGN KEY (scheme_id) REFERENCES dim_scheme(scheme_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);

"""

cursor.executescript(schema_sql)

conn.commit()

print("Star schema created successfully.")

Star schema created successfully.


In [4]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = cursor.execute(query).fetchall()

print(tables)

[('nav_history_cleaned',), ('investor_transactions_cleaned',), ('scheme_performance_cleaned',), ('dim_scheme',), ('dim_investor',), ('dim_date',), ('fact_transactions',), ('fact_nav_history',), ('fact_scheme_performance',)]


In [5]:
import pandas as pd
from sqlalchemy import create_engine, text

nav_df = pd.read_csv("/content/nav_history_cleaned.csv")
transactions_df = pd.read_csv("/content/investor_transactions_cleaned.csv")
performance_df = pd.read_csv("/content/scheme_performance_cleaned.csv")

engine = create_engine("sqlite:///mutual_fund_dw.db")

nav_df.to_sql(
    "nav_history_cleaned",
    con=engine,
    if_exists="replace",
    index=False
)

transactions_df.to_sql(
    "investor_transactions_cleaned",
    con=engine,
    if_exists="replace",
    index=False
)

performance_df.to_sql(
    "scheme_performance_cleaned",
    con=engine,
    if_exists="replace",
    index=False
)

print("Datasets loaded successfully into SQLite.\n")

with engine.connect() as conn:
    nav_sql_count = conn.execute(
        text("SELECT COUNT(*) FROM nav_history_cleaned")
    ).scalar()

    transactions_sql_count = conn.execute(
        text("SELECT COUNT(*) FROM investor_transactions_cleaned")
    ).scalar()

    performance_sql_count = conn.execute(
        text("SELECT COUNT(*) FROM scheme_performance_cleaned")
    ).scalar()

print("Row Count Verification")
print("-" * 40)

print(f"NAV CSV Rows              : {len(nav_df)}")
print(f"NAV SQLite Rows           : {nav_sql_count}\n")

print(f"Transactions CSV Rows     : {len(transactions_df)}")
print(f"Transactions SQLite Rows  : {transactions_sql_count}\n")

print(f"Performance CSV Rows      : {len(performance_df)}")
print(f"Performance SQLite Rows   : {performance_sql_count}\n")

assert len(nav_df) == nav_sql_count
assert len(transactions_df) == transactions_sql_count
assert len(performance_df) == performance_sql_count

print("All row counts match successfully.")

Datasets loaded successfully into SQLite.

Row Count Verification
----------------------------------------
NAV CSV Rows              : 46000
NAV SQLite Rows           : 46000

Transactions CSV Rows     : 32778
Transactions SQLite Rows  : 32778

Performance CSV Rows      : 40
Performance SQLite Rows   : 40

All row counts match successfully.


Queries

In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_dw.db")

query = """
PRAGMA table_info(scheme_performance_cleaned);
"""

columns = pd.read_sql(query, conn)

print(columns)

conn.close()

    cid                name    type  notnull dflt_value  pk
0     0           amfi_code  BIGINT        0       None   0
1     1         scheme_name    TEXT        0       None   0
2     2          fund_house    TEXT        0       None   0
3     3            category    TEXT        0       None   0
4     4                plan    TEXT        0       None   0
5     5      return_1yr_pct   FLOAT        0       None   0
6     6      return_3yr_pct   FLOAT        0       None   0
7     7      return_5yr_pct   FLOAT        0       None   0
8     8   benchmark_3yr_pct   FLOAT        0       None   0
9     9               alpha   FLOAT        0       None   0
10   10                beta   FLOAT        0       None   0
11   11        sharpe_ratio   FLOAT        0       None   0
12   12       sortino_ratio   FLOAT        0       None   0
13   13     std_dev_ann_pct   FLOAT        0       None   0
14   14    max_drawdown_pct   FLOAT        0       None   0
15   15           aum_crore  BIGINT     

In [15]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_dw.db")

# Get schema for nav_history_cleaned
nav_history_schema = pd.read_sql("PRAGMA table_info(nav_history_cleaned);", conn)

# Get schema for investor_transactions_cleaned
investor_transactions_schema = pd.read_sql("PRAGMA table_info(investor_transactions_cleaned);", conn)

conn.close()

print("Schema for nav_history_cleaned:")
print(nav_history_schema)
print("\nSchema for investor_transactions_cleaned:")
print(investor_transactions_schema)

Schema for nav_history_cleaned:
   cid       name    type  notnull dflt_value  pk
0    0  amfi_code  BIGINT        0       None   0
1    1       date    TEXT        0       None   0
2    2        nav   FLOAT        0       None   0

Schema for investor_transactions_cleaned:
    cid                name    type  notnull dflt_value  pk
0     0         investor_id    TEXT        0       None   0
1     1    transaction_date    TEXT        0       None   0
2     2           amfi_code  BIGINT        0       None   0
3     3    transaction_type    TEXT        0       None   0
4     4          amount_inr  BIGINT        0       None   0
5     5               state    TEXT        0       None   0
6     6                city    TEXT        0       None   0
7     7           city_tier    TEXT        0       None   0
8     8           age_group    TEXT        0       None   0
9     9              gender    TEXT        0       None   0
10   10  annual_income_lakh   FLOAT        0       None   0
11   

In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("mutual_fund_dw.db")

# 1. Top 5 Funds by AUM
query1 = """
SELECT
    amfi_code,
    scheme_name,
    aum_crore
FROM scheme_performance_cleaned
ORDER BY aum_crore DESC
LIMIT 5;
"""

print("\nTop 5 Funds by AUM")
print(pd.read_sql(query1, conn))


# 2. Average NAV Per Month
query2 = """
SELECT
    strftime('%Y', date) AS year,
    strftime('%m', date) AS month,
    AVG(nav) AS avg_monthly_nav
FROM nav_history_cleaned
GROUP BY year, month
ORDER BY year, month;
"""

print("\nAverage NAV Per Month")
print(pd.read_sql(query2, conn))


# 3. SIP YoY Growth
query3 = """
SELECT
    strftime('%Y', transaction_date) AS year,
    SUM(amount_inr) AS total_sip_amount
FROM investor_transactions_cleaned
WHERE LOWER(transaction_type) = 'sip'
GROUP BY year
ORDER BY year;
"""

print("\nSIP YoY Growth")
print(pd.read_sql(query3, conn))


# 4. Transactions by State
query4 = """
SELECT
    state,
    COUNT(*) AS total_transactions,
    SUM(amount_inr) AS total_amount
FROM investor_transactions_cleaned
GROUP BY state
ORDER BY total_amount DESC;
"""

print("\nTransactions by State")
print(pd.read_sql(query4, conn))


# 5. Funds with Expense Ratio < 1%
query5 = """
SELECT
    amfi_code,
    scheme_name,
    expense_ratio_pct
FROM scheme_performance_cleaned
WHERE expense_ratio_pct < 1;
"""

print("\nFunds with Expense Ratio < 1%")
print(pd.read_sql(query5, conn))


# 6. Top 10 Investors by Investment Amount
query6 = """
SELECT
    it.investor_id,
    di.investor_name,
    SUM(it.amount_inr) AS total_investment
FROM investor_transactions_cleaned AS it
JOIN dim_investor AS di ON it.investor_id = di.investor_id
GROUP BY it.investor_id, di.investor_name
ORDER BY total_investment DESC
LIMIT 10;
"""

print("\nTop 10 Investors by Investment Amount")
print(pd.read_sql(query6, conn))


# 7. Fund Category Wise Average Returns
query7 = """
SELECT
    category,
    AVG(return_1yr_pct) AS avg_1yr_return,
    AVG(return_3yr_pct) AS avg_3yr_return,
    AVG(return_5yr_pct) AS avg_5yr_return
FROM scheme_performance_cleaned
GROUP BY category
ORDER BY avg_5yr_return DESC;
"""

print("\nFund Category Wise Average Returns")
print(pd.read_sql(query7, conn))


# 8. Monthly Transaction Volume Trend
query8 = """
SELECT
    strftime('%Y', transaction_date) AS year,
    strftime('%m', transaction_date) AS month,
    COUNT(*) AS transaction_count,
    SUM(amount_inr) AS transaction_volume
FROM investor_transactions_cleaned
GROUP BY year, month
ORDER BY year, month;
"""

print("\nMonthly Transaction Volume Trend")
print(pd.read_sql(query8, conn))


# 9. Best Performing Funds Compared to Benchmark
query9 = """
SELECT
    amfi_code,
    scheme_name,
    return_5yr_pct,
    benchmark_3yr_pct,
    (return_5yr_pct - benchmark_3yr_pct) AS excess_return
FROM scheme_performance_cleaned
ORDER BY excess_return DESC
LIMIT 10;
"""

print("\nBest Performing Funds Compared to Benchmark")
print(pd.read_sql(query9, conn))


# 10. Most Popular Transaction Type
query10 = """
SELECT
    transaction_type,
    COUNT(*) AS total_transactions,
    SUM(amount_inr) AS total_amount
FROM investor_transactions_cleaned
GROUP BY transaction_type
ORDER BY total_transactions DESC;
"""

print("\nMost Popular Transaction Type")
print(pd.read_sql(query10, conn))

conn.close()


Top 5 Funds by AUM
   amfi_code                                        scheme_name  aum_crore
0     148568  Mirae Asset Emerging Bluechip Fund - Regular -...      49046
1     120842      Kotak Emerging Equity Fund - Regular - Growth      47469
2     118634     Nippon India Small Cap Fund - Regular - Growth      43630
3     149322         DSP Top 100 Equity Fund - Regular - Growth      41828
4     102886                UTI Mid Cap Fund - Regular - Growth      41728

Average NAV Per Month
    year month  avg_monthly_nav
0   2022    01       207.061394
1   2022    02       207.717759
2   2022    03       209.692614
3   2022    04       211.833457
4   2022    05       212.731451
5   2022    06       213.860867
6   2022    07       213.956111
7   2022    08       215.683994
8   2022    09       218.494307
9   2022    10       219.529633
10  2022    11       223.470692
11  2022    12       226.760632
12  2023    01       230.671234
13  2023    02       233.847674
14  2023    03       238.00

Load all cleaned datasets into SQLite — use SQLAlchemy create_engine + df.to_sql(). Verify row counts match source CSVs.

In [18]:
import os
import pandas as pd
from sqlalchemy import create_engine, text



files = {
    'nav_history': 'nav_history_cleaned.csv',
    'investor_transactions': 'investor_transactions_cleaned.csv',
    'scheme_performance': 'scheme_performance_cleaned.csv',
    'fund_master': 'cleaned_01_fund_master.csv',
    'aum_by_fund_house': 'cleaned_03_aum_by_fund_house.csv',
    'monthly_sip_inflows': 'cleaned_04_monthly_sip_inflows.csv',
    'category_inflows': 'cleaned_05_category_inflows.csv',
    'industry_folio_count': 'cleaned_06_industry_folio_count.csv',
    'portfolio_holdings': 'cleaned_09_portfolio_holdings.csv',
    'benchmark_indices': 'cleaned_10_benchmark_indices.csv'
}



engine = create_engine('sqlite:///mutual_funds.db')

results = []

print("Starting data ingestion into SQLite...\n")

for table_name, csv_file in files.items():
    if not os.path.exists(csv_file):
        print(f"Warning: File {csv_file} not found in the current directory. Skipping...")
        continue



    df = pd.read_csv(csv_file)
    csv_row_count = len(df)



    df.to_sql(table_name, engine, if_exists='replace', index=False)



    with engine.connect() as connection:
        query = text(f"SELECT COUNT(*) FROM {table_name}")
        sql_row_count = connection.execute(query).scalar()

    matches = (csv_row_count == sql_row_count)

    results.append({
        'Table Name': table_name,
        'CSV Rows': csv_row_count,
        'SQL Rows': sql_row_count,
        'Match Verification': "Passed" if matches else "Failed"
    })

summary_df = pd.DataFrame(results)
print("\nVERIFICATION SUMMARY")
print(summary_df.to_string(index=False))

Starting data ingestion into SQLite...


VERIFICATION SUMMARY
           Table Name  CSV Rows  SQL Rows Match Verification
          nav_history     46000     46000             Passed
investor_transactions     32778     32778             Passed
   scheme_performance        40        40             Passed
          fund_master        40        40             Passed
    aum_by_fund_house        90        90             Passed
  monthly_sip_inflows        48        48             Passed
     category_inflows       144       144             Passed
 industry_folio_count        21        21             Passed
   portfolio_holdings       322       322             Passed
    benchmark_indices      8050      8050             Passed


Write 10 analytical SQL queries — top 5 funds by AUM, average NAV per month, SIP YoY growth, transactions by state, funds with expense_ratio < 1%, and 5 more of your choice.

In [20]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///mutual_funds.db')

queries = {
    "1. Top 5 Funds by AUM": """
        SELECT amfi_code, scheme_name, category, aum_crore
        FROM scheme_performance
        ORDER BY aum_crore DESC
        LIMIT 5;
    """,

    "2. Average NAV Per Month (Sample)": """
        SELECT amfi_code, strftime('%Y-%m', date) AS month, ROUND(AVG(nav), 4) AS avg_nav
        FROM nav_history
        GROUP BY amfi_code, month
        ORDER BY amfi_code, month
        LIMIT 10;
    """,

    "3. SIP Inflows Year-over-Year (YoY) Growth": """
        WITH yearly_inflow AS (
            SELECT strftime('%Y', month) AS year, SUM(sip_inflow_crore) AS total_sip_inflow
            FROM monthly_sip_inflows
            GROUP BY year
        )
        SELECT
            y1.year,
            ROUND(y1.total_sip_inflow, 2) AS current_year_inflow_cr,
            ROUND(y2.total_sip_inflow, 2) AS prev_year_inflow_cr,
            ROUND(((1.0 * y1.total_sip_inflow - y2.total_sip_inflow) / y2.total_sip_inflow) * 100, 2) AS yoy_growth_pct
        FROM yearly_inflow y1
        LEFT JOIN yearly_inflow y2 ON CAST(y1.year AS INTEGER) = CAST(y2.year AS INTEGER) + 1;
    """,

    "4. Total Transactions and Capital Invested by State": """
        SELECT state, COUNT(*) AS total_transactions, SUM(amount_inr) AS total_amount_inr
        FROM investor_transactions
        GROUP BY state
        ORDER BY total_amount_inr DESC;
    """,

    "5. Highly Cost-Effective Funds (Expense Ratio < 1%)": """
        SELECT amfi_code, scheme_name, category, expense_ratio_pct
        FROM scheme_performance
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC;
    """,

    "6. Top 5 Sectors by Total Portfolio Market Value": """
        SELECT sector, ROUND(SUM(market_value_cr), 2) AS total_market_value_cr, ROUND(AVG(weight_pct), 2) AS avg_allocation_pct
        FROM portfolio_holdings
        GROUP BY sector
        ORDER BY total_market_value_cr DESC
        LIMIT 5;
    """,

    "7. Volume & Amounts by Transaction Type (SIP vs Lumpsum vs Redemption)": """
        SELECT transaction_type, COUNT(*) AS txn_count, SUM(amount_inr) AS total_amount_inr
        FROM investor_transactions
        GROUP BY transaction_type
        ORDER BY txn_count DESC;
    """,

    "8. Top 5 Funds with the Highest Alpha (Outperformance)": """
        SELECT amfi_code, scheme_name, category, alpha, sharpe_ratio
        FROM scheme_performance
        ORDER BY alpha DESC
        LIMIT 5;
    """,

    "9. Demographic Capital Breakdown (By Age Group & Gender)": """
        SELECT age_group, gender, COUNT(*) AS txn_count, SUM(amount_inr) AS total_amount_inr
        FROM investor_transactions
        GROUP BY age_group, gender
        ORDER BY total_amount_inr DESC;
    """,

    "10. Historical AUM Trends by Fund House (Sample)": """
        SELECT date, fund_house, aum_crore
        FROM aum_by_fund_house
        ORDER BY fund_house, date
        LIMIT 10;
    """
}

print("Executing 10 Analytical SQL Queries...\n" + "="*60)

with engine.connect() as connection:
    for title, query_sql in queries.items():
        print(f"\n🔹 {title}")
        print("-" * len(title))

        df_result = pd.read_sql_query(text(query_sql), connection)

        if df_result.empty:
            print("No data returned.")
        else:
            print(df_result.to_string(index=False))
        print("="*60)

Executing 10 Analytical SQL Queries...

🔹 1. Top 5 Funds by AUM
---------------------
 amfi_code                                           scheme_name        category  aum_crore
    148568 Mirae Asset Emerging Bluechip Fund - Regular - Growth Large & Mid Cap      49046
    120842         Kotak Emerging Equity Fund - Regular - Growth         Mid Cap      47469
    118634        Nippon India Small Cap Fund - Regular - Growth       Small Cap      43630
    149322            DSP Top 100 Equity Fund - Regular - Growth       Large Cap      41828
    102886                   UTI Mid Cap Fund - Regular - Growth         Mid Cap      41728

🔹 2. Average NAV Per Month (Sample)
---------------------------------
 amfi_code   month  avg_nav
    100016 2022-01 512.5353
    100016 2022-02 513.9306
    100016 2022-03 522.5782
    100016 2022-04 525.6312
    100016 2022-05 504.3125
    100016 2022-06 465.1370
    100016 2022-07 436.7460
    100016 2022-08 421.3311
    100016 2022-09 422.1759
    100016 